# Lab 05: Single Agent with LangChain

## Overview

In this lab you will build a **production-ready single agent** that combines multiple tools, classifies user intent to route queries intelligently, wraps everything in the **Databricks `ChatAgent` interface** for Model Serving compatibility, and logs the result to **MLflow** for registration in Unity Catalog.

| Attribute | Detail |
|---|---|
| **Exam Domain** | Application Development (30%) |
| **Key Skills** | Multi-tool agents, intent classification routing, ChatAgent interface, MLflow model logging |
| **Estimated Time** | 35 minutes |
| **Estimated Cost** | $1–2 |

### What You Will Build

```
User Input
    └► Intent Classifier (classify: RESEARCH / CITATION / METADATA)
            ├► RESEARCH  → Retrieval Agent (Vector Search)
            ├► CITATION  → format_citation UC Function
            └► METADATA  → get_paper_metadata UC Function
                    └► ChatAgent Wrapper (MLflow-compatible)
                            └► Model Serving endpoint
```

Building on the tools from Labs 03 and 04, this lab adds the intent-routing layer and the `ChatAgent` interface that Databricks Model Serving expects.

In [ ]:
# Prerequisites — run once per session
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-community unitycatalog-langchain databricks-agents --quiet
dbutils.library.restartPython()

> **Cost tip:** These labs run on **serverless compute** by default — no cluster setup needed. You only pay per-second of actual usage. The `COST_PROFILE` below lets you choose cheaper models if you're cost-sensitive.

In [ ]:
# === Cost Profile ===
# "budget"   — cheapest models, may have lower quality (~$0.50/lab)
# "balanced" — good quality/cost ratio (~$1-2/lab)  [DEFAULT]
# "quality"  — best models, highest cost (~$3-5/lab)
COST_PROFILE = "balanced"

_LLM_ENDPOINTS = {
    "budget":   "databricks-meta-llama-3-1-8b-instruct",
    "balanced": "databricks-meta-llama-3-3-70b-instruct",
    "quality":  "databricks-meta-llama-3-3-70b-instruct",
}
LLM_ENDPOINT = _LLM_ENDPOINTS[COST_PROFILE]

# ---------------------------------------------------------------------------
# Configuration — update these values to match your environment
# ---------------------------------------------------------------------------
CATALOG      = "genai_lab_guide"               # Unity Catalog catalog name
SCHEMA       = "default"                        # Schema inside the catalog
VS_ENDPOINT  = "genai_lab_guide_vs_endpoint"   # Vector Search endpoint name
VS_INDEX     = f"{CATALOG}.{SCHEMA}.arxiv_chunks_index"

print(f"Catalog   : {CATALOG}")
print(f"Schema    : {SCHEMA}")
print(f"VS Index  : {VS_INDEX}")

# ---------------------------------------------------------------------------
# Rebuild tools from Labs 03 & 04
# These tools were created in previous labs; we reconstruct them here so this
# notebook runs standalone.  See lab 03 (retrieval tool) and lab 04 (UC tools).
# ---------------------------------------------------------------------------
from databricks.vector_search.client import VectorSearchClient
from langchain_community.chat_models import ChatDatabricks
from langchain_core.tools import tool
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from unitycatalog.ai.langchain.toolkit import UCFunctionToolkit

# ── Retrieval tool (Lab 03) ────────────────────────────────────────────────────────
vsc   = VectorSearchClient()
index = vsc.get_index(VS_ENDPOINT, VS_INDEX)


def retrieve_context(query: str, num_results: int = 3) -> str:
    results = index.similarity_search(
        query_text=query,
        columns=["chunk_text", "path"],
        num_results=num_results,
        query_type="HYBRID",
    )
    docs = results.get("result", {}).get("data_array", [])
    parts = []
    for doc in docs:
        source = doc[1].split("/")[-1].replace(".pdf", "")
        parts.append(f"[Source: {source}]\n{doc[0]}")
    return "\n\n---\n\n".join(parts)


@tool
def search_arxiv_papers(query: str) -> str:
    """Search the arXiv paper corpus for context relevant to the user's question.
    Always call this tool before answering research questions."""
    return retrieve_context(query)


# ── UC function tools (Lab 04) ────────────────────────────────────────────────
client = DatabricksFunctionClient()
uc_toolkit = UCFunctionToolkit(
    function_names=[
        f"{CATALOG}.{SCHEMA}.get_paper_metadata",
        f"{CATALOG}.{SCHEMA}.format_citation",
    ],
    client=client,
)
uc_tools = uc_toolkit.tools

# ── LLM ─────────────────────────────────────────────────────────────────────────
llm = ChatDatabricks(
    endpoint=LLM_ENDPOINT,
    max_tokens=1024,
    temperature=0.1,
)

print(f"UC tools : {[t.name for t in uc_tools]}")
print("All tools rebuilt successfully.")

## A. Multi-Tool Agent

The first step is to assemble a single `AgentExecutor` that can use all three tools:

- **`search_arxiv_papers`** — Vector Search retrieval (Lab 03)
- **`get_paper_metadata`** — UC SQL function returning chunk statistics (Lab 04)
- **`format_citation`** — UC Python function returning APA citation strings (Lab 04)

The LLM reads each tool’s name and description to decide which tool(s) to call, then synthesises a final response from the tool outputs. A single agent with all tools is the correct starting point before adding an intent-routing layer on top.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor

# ── Combine all tools ────────────────────────────────────────────────────────────
all_tools = [search_arxiv_papers] + uc_tools
print(f"Tools: {[t.name for t in all_tools]}")

# ── System prompt ──────────────────────────────────────────────────────────────────
multi_tool_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an arXiv Research Assistant with access to three tools:\n"
        "- search_arxiv_papers: retrieve relevant passages from the paper corpus\n"
        "- get_paper_metadata: look up chunk statistics for a given paper\n"
        "- format_citation: generate APA-style citation strings\n"
        "Select the most appropriate tool(s) for each question.",
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# ── Build agent ────────────────────────────────────────────────────────────────────────
agent          = create_tool_calling_agent(llm, all_tools, multi_tool_prompt)
agent_executor = AgentExecutor(agent=agent, tools=all_tools, verbose=True)

# ── Test with a diverse query ──────────────────────────────────────────────────
diverse_query = (
    "Explain what LoRA is, tell me how many chunks we have for that paper, "
    "and format a citation for Hu et al. 2021 arXiv:2106.09685."
)

result = agent_executor.invoke({"input": diverse_query})
print("\n" + "=" * 70)
print(result["output"])

## B. Intent Classification Workflow

While a single agent works, routing requests explicitly gives you:

- **Lower latency** — the classifier steers the LLM to use only the relevant tool(s), reducing unnecessary tool calls.
- **Better control** — you can tailor the system prompt and tool subset to each intent class.
- **Observability** — MLflow traces show classified intent alongside tool calls.

The three intent classes are:

| Intent | Description | Tools Invoked |
|---|---|---|
| `RESEARCH` | Content or concept questions | `search_arxiv_papers` |
| `CITATION` | Format a bibliographic reference | `format_citation` |
| `METADATA` | Paper statistics or chunk counts | `get_paper_metadata` |

The workflow is: **classify intent → build tailored prompt → invoke agent**.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Classifier chain ─────────────────────────────────────────────────────────────
classifier_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an intent classifier for a research assistant.\n"
        "Classify the user's query into exactly one of: RESEARCH, CITATION, METADATA.\n\n"
        "RESEARCH  — the user wants explanations, summaries, or conceptual content from papers.\n"
        "CITATION  — the user wants a formatted bibliographic reference.\n"
        "METADATA  — the user wants statistics about a paper (e.g. chunk count, file path).\n\n"
        "Respond with only the label — no punctuation, no explanation.",
    ),
    ("human", "{query}"),
])

classifier_chain = classifier_prompt | llm | StrOutputParser()


# ── Intent-specific system prompts ──────────────────────────────────────────────
INTENT_PROMPTS = {
    "RESEARCH": (
        "You are an arXiv Research Assistant. "
        "Always retrieve relevant passages with search_arxiv_papers before answering. "
        "Ground your response in the retrieved context and cite sources."
    ),
    "CITATION": (
        "You are a citation formatter. "
        "Use the format_citation tool to produce an APA-style reference. "
        "Extract author, title, year, and arXiv ID from the user's query."
    ),
    "METADATA": (
        "You are a data assistant. "
        "Use get_paper_metadata to look up chunk statistics for the requested paper. "
        "Report the path and chunk count clearly."
    ),
}


def route_query(query: str) -> dict:
    """Classify the query intent, then invoke the agent with a tailored prompt."""
    # Step 1: classify
    intent = classifier_chain.invoke({"query": query}).strip().upper()
    if intent not in INTENT_PROMPTS:
        intent = "RESEARCH"  # safe fallback

    print(f"Classified intent: {intent}")

    # Step 2: build intent-specific prompt
    routed_prompt = ChatPromptTemplate.from_messages([
        ("system", INTENT_PROMPTS[intent]),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])

    # Step 3: build and invoke agent
    routed_agent    = create_tool_calling_agent(llm, all_tools, routed_prompt)
    routed_executor = AgentExecutor(agent=routed_agent, tools=all_tools, verbose=False)
    result          = routed_executor.invoke({"input": query})

    return {"intent": intent, "output": result["output"]}


# ── Test each intent ─────────────────────────────────────────────────────────────────
test_cases = [
    ("RESEARCH",  "What is the key idea behind the transformer attention mechanism?"),
    ("CITATION",  "Format a citation for Devlin et al. 2018, BERT, arXiv:1810.04805."),
    ("METADATA",  "How many chunks do we have for the LoRA paper?"),
]

for expected_intent, query in test_cases:
    print("=" * 70)
    print(f"Query ({expected_intent}): {query}")
    response = route_query(query)
    print(f"Detected: {response['intent']} | Answer: {response['output'][:200]}")
    print()

## C. ChatAgent Interface

Databricks Model Serving expects models to conform to the **`ChatAgent` interface** from `databricks.agents`. This ensures:

- Consistent request/response schema (OpenAI-compatible message format).
- Automatic tracing integration with MLflow.
- Compatibility with the Databricks AI Gateway and Review App.

A `ChatAgent` subclass must implement **`predict(messages, context=None)`**, which:
1. Receives a list of `ChatAgentMessage` objects.
2. Returns a `ChatAgentResponse` wrapping the assistant reply.

The `predict()` method is the single required interface point — everything else (tool setup, LLM config) lives inside the class.

In [ ]:
from databricks.agents import ChatAgent, ChatAgentMessage, ChatAgentResponse
from typing import Optional


class ArxivResearchAgent(ChatAgent):
    """
    A Databricks ChatAgent that classifies user intent and routes queries
    to the appropriate tool via a LangChain AgentExecutor.

    Inheriting from ChatAgent makes this class compatible with:
    - mlflow.pyfunc.log_model (via the ChatAgent flavour)
    - Databricks Model Serving (REST endpoint)
    - Databricks AI Gateway and Review App
    """

    def __init__(self):
        # Rebuild tools and executor at initialisation time so the object is
        # self-contained when loaded from an MLflow artifact.
        self._vsc   = VectorSearchClient()
        self._index = self._vsc.get_index(VS_ENDPOINT, VS_INDEX)
        self._llm   = ChatDatabricks(
            endpoint=LLM_ENDPOINT,
            max_tokens=1024,
            temperature=0.1,
        )
        self._all_tools = [search_arxiv_papers] + uc_tools
        self._classifier = classifier_chain  # reuse chain from Section B

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[dict] = None,
    ) -> ChatAgentResponse:
        """
        Required ChatAgent method.

        Parameters
        ----------
        messages : list[ChatAgentMessage]
            Conversation history.  The last message is the current user turn.
        context : dict, optional
            Unused; reserved for future Databricks context injection.

        Returns
        -------
        ChatAgentResponse
            Wraps the assistant's reply in the standard response envelope.
        """
        # Extract the latest user message
        user_query = messages[-1].content if messages else ""

        # Classify intent
        intent = self._classifier.invoke({"query": user_query}).strip().upper()
        if intent not in INTENT_PROMPTS:
            intent = "RESEARCH"

        # Build intent-specific prompt and executor
        routed_prompt = ChatPromptTemplate.from_messages([
            ("system", INTENT_PROMPTS[intent]),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad"),
        ])
        routed_agent    = create_tool_calling_agent(
            self._llm, self._all_tools, routed_prompt
        )
        routed_executor = AgentExecutor(
            agent=routed_agent, tools=self._all_tools, verbose=False
        )

        result = routed_executor.invoke({"input": user_query})

        return ChatAgentResponse(
            messages=[
                ChatAgentMessage(role="assistant", content=result["output"])
            ]
        )


# ── Smoke-test the class ───────────────────────────────────────────────────────────────
agent_obj = ArxivResearchAgent()

test_messages = [
    ChatAgentMessage(role="user", content="Summarise the key idea behind LoRA."),
]

response = agent_obj.predict(test_messages)
print("ChatAgent response:")
print(response.messages[0].content)

## D. Log and Register ChatAgent

Logging the `ChatAgent` with `mlflow.pyfunc.log_model` captures:

- The Python class and all its dependencies.
- The model signature (OpenAI-compatible chat format).
- Pip requirements for reproducible serving.

After logging, `mlflow.register_model` promotes the artifact to the Unity Catalog model registry, where it can be deployed as a real-time Model Serving endpoint in one click.

In [ ]:
import mlflow
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

username = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{username}/genai-lab-guide/lab-05-single-agent-langchain")

# ── Model signature: OpenAI-compatible chat format ───────────────────────────────
input_schema  = Schema([ColSpec("string", "messages")])
output_schema = Schema([ColSpec("string", "messages")])
signature     = ModelSignature(inputs=input_schema, outputs=output_schema)

# ── Log the ChatAgent ──────────────────────────────────────────────────────────────
with mlflow.start_run(run_name="arxiv-chat-agent-v1") as run:
    mlflow.log_params({
        "llm_endpoint"  : LLM_ENDPOINT,
        "intent_classes": "RESEARCH,CITATION,METADATA",
        "vs_index"      : VS_INDEX,
        "tool_count"    : len(all_tools),
    })

    model_info = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model=ArxivResearchAgent(),
        signature=signature,
        input_example={
            "messages": [{"role": "user", "content": "What is attention?"}]
        },
        pip_requirements=[
            "databricks-sdk",
            "databricks-vectorsearch",
            "mlflow",
            "langchain",
            "langchain-community",
            "unitycatalog-langchain",
            "databricks-agents",
        ],
    )

    run_id = run.info.run_id

print(f"Run ID    : {run_id}")
print(f"Model URI : {model_info.model_uri}")

# ── Register in Unity Catalog ────────────────────────────────────────────────────
mlflow.set_registry_uri("databricks-uc")

UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.arxiv_chat_agent"

registered = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=UC_MODEL_NAME,
)

print(f"\nRegistered model : {registered.name}")
print(f"Version          : {registered.version}")
print(f"Status           : {registered.status}")

## Key Takeaways

| Concept | What You Did |
|---|---|
| **Multi-Tool Agent** | Combined retrieval, SQL metadata, and Python citation tools in a single `AgentExecutor` |
| **Intent Routing** | Built a classifier chain that steers queries to the correct tool with a tailored system prompt |
| **ChatAgent Interface** | Wrapped the routed agent in `ArxivResearchAgent(ChatAgent)` for Model Serving compatibility |
| **MLflow Registration** | Logged the `ChatAgent` with `mlflow.pyfunc.log_model` and registered it in Unity Catalog |

### Exam Tips

- `ChatAgent.predict(messages, context)` is the **required** method signature — the exam tests this frequently.
- `ChatAgentResponse` must wrap replies in a list of `ChatAgentMessage` objects with `role="assistant"`.
- Intent classification before tool dispatch reduces unnecessary LLM tool calls and improves latency.
- Use `mlflow.set_registry_uri("databricks-uc")` **before** `mlflow.register_model` to target Unity Catalog.
- The `"databricks_agents"` flavour in `log_model` ensures the Chat UI and Review App can load the model correctly.

## Architecture Diagram

![Architecture Diagram](../../assets/diagrams/lab-05-single-agent-langchain.png)

## Key Concepts

| Concept | Definition |
|---|---|
| **Multi-Step Workflow** | An agent pipeline where the output of one step (e.g. intent classification) controls the behaviour of the next step (e.g. tool selection and prompt construction) |
| **Intent Classification** | A lightweight LLM chain that maps free-text user input to a discrete label, used to route queries to specialised sub-agents or tool subsets |
| **ChatAgent** | Databricks base class (`databricks.agents.ChatAgent`) for building Model Serving-compatible agents; requires implementing `predict(messages, context)` |
| **Tool Calling Agent** | A LangChain agent pattern (`create_tool_calling_agent`) where the LLM emits structured tool-call requests and the `AgentExecutor` handles the observe-act loop |
| **AgentExecutor** | LangChain runtime that manages the tool-calling loop: invoke agent → parse tool calls → execute tools → feed observations back → repeat until final answer |
| **ChatAgentResponse** | The return type of `ChatAgent.predict()` — a wrapper containing a list of `ChatAgentMessage` objects conforming to the OpenAI message schema |
| **Model Signature** | The input/output schema attached to an MLflow model; required for Model Serving validation and automatic request/response schema enforcement |

## Exam Preparation

### Exam Practice Questions

**Q1.** A developer wants to route user queries to different tool subsets depending on whether the query is a content question, a citation request, or a metadata lookup. Which pattern best describes this approach?

- A) Single-agent with all tools loaded simultaneously
- B) Multi-step workflow with intent classification routing
- C) Multi-agent orchestration with a supervisor agent
- D) Retrieval-augmented generation without tool calling

**Answer: B** — Classifying intent before invoking the agent is a multi-step workflow. The classifier output controls which tools and system prompt are used in the next step. Single-agent (A) loads all tools but does not pre-route. Multi-agent (C) involves separate agents, not a classifier. RAG alone (D) does not address routing.

---

**Q2.** What is the key difference between `AgentExecutor` and `ChatAgent` in the Databricks ecosystem?

- A) `AgentExecutor` is for multi-turn conversations; `ChatAgent` is for single-turn only
- B) `AgentExecutor` manages the tool-calling loop at runtime; `ChatAgent` is a Databricks interface class that wraps an agent for Model Serving compatibility
- C) `AgentExecutor` requires Unity Catalog tools; `ChatAgent` supports any LangChain tool
- D) `ChatAgent` automatically handles multi-agent orchestration; `AgentExecutor` does not

**Answer: B** — `AgentExecutor` is the LangChain runtime that executes the observe-act loop. `ChatAgent` is a Databricks interface class whose `predict()` method makes any agent compatible with Model Serving, the Review App, and the AI Gateway. They are complementary: `AgentExecutor` runs inside the `ChatAgent.predict()` method.

---

**Q3.** When should you use a multi-agent architecture instead of a single agent with intent routing?

- A) Whenever you have more than two intent classes
- B) When the agent needs to call more than three tools
- C) When sub-tasks require genuinely independent reasoning loops, specialised prompts, or different LLMs that cannot be collapsed into a single `AgentExecutor`
- D) When deploying to Model Serving, which requires one agent per endpoint

**Answer: C** — Multi-agent is appropriate when sub-tasks have fundamentally different requirements (different LLMs, isolated tool sets, parallel execution) or when a single agent's context window cannot fit all required tools and instructions. Simple intent routing within a single agent is sufficient for most cases with overlapping tools.

---

**Q4.** What is the required method signature for a class that inherits from `ChatAgent`?

- A) `def run(self, query: str) -> str`
- B) `def invoke(self, inputs: dict) -> dict`
- C) `def predict(self, messages: list[ChatAgentMessage], context: Optional[dict] = None) -> ChatAgentResponse`
- D) `def generate(self, prompt: str, max_tokens: int) -> str`

**Answer: C** — The `ChatAgent` interface requires exactly `predict(self, messages, context=None) -> ChatAgentResponse`. The exam tests this signature directly. Any other method name or signature will cause Model Serving deployment to fail.

---

**Q5.** An intent classifier returns `RESEARCH` for 80% of queries, `CITATION` for 15%, and `METADATA` for 5%. A developer proposes removing the classifier and letting the single agent select tools freely. What is the main trade-off?

- A) Without the classifier, the agent cannot use UC function tools
- B) Without the classifier, the agent may make speculative tool calls for CITATION and METADATA queries that do not require retrieval, increasing latency and token cost
- C) Without the classifier, MLflow tracing stops working
- D) Without the classifier, the ChatAgent interface is no longer valid

**Answer: B** — A single agent without a classifier may call multiple tools to "cover its bases," especially for ambiguous queries. The classifier eliminates speculative calls by constraining the agent's tool selection before the reasoning loop begins, reducing latency and cost. Options A, C, and D are incorrect — tool availability, MLflow tracing, and the ChatAgent interface are independent of whether a classifier is present.

## Cost Breakdown

| Component | Detail | Estimated Cost |
|---|---|---|
| Databricks Serverless compute | Notebook execution + UC function calls (~20 min DBU) | ~$0.50 |
| LLM token usage | Classifier call + routed agent call per test query (6 total calls) | ~$0.50 |
| Vector Search queries | Retrieval calls during RESEARCH intent tests | Included in serverless |
| **Total** | | **~$1–2** |

**Estimated time:** ~35 min | **Estimated cost:** ~$1–2

> Costs vary by workspace region and current DBU pricing. Use the Databricks Cost Dashboard to track actuals.